<img src="https://raw.githubusercontent.com/drdave-teaching/OPIM5641-notebooks/main/_banners/opim5641_banner.svg" width="100%" alt="OPIM 5641 banner"/>

**Dr. Dave Wanik - Operations and Information Management - University of Connecticut**

---

# Studio 3.1 — Pyomo Basics: Hard-Coded vs. Soft-Coded Models

So far we've solved LPs three ways: brute force, the graphical method, and Simplex by hand. Tonight
we hand the work to a solver. You write down the model, **Pyomo** builds it, and the solver does the
Simplex for you.

We start with a problem you already know the answer to: **Flair's Furniture (320 tables, 360 chairs,
\$4,040)**. If the solver agrees, the code is right.

We will write it two ways:

- **Version A, hard-coded:** every number from the word problem is typed straight into the model.
- **Version B, soft-coded:** the numbers live in dictionaries, and the model reads them with loops.

**How this notebook works.** All of the code is already here. You do not have to write a model
from scratch. Your job is to **run it, read it, and change it**: each experiment has a clearly marked
cell where you plug in a scenario, re-run, and see what happens.

## Import modules (download Pyomo)

Run this first, every time. It installs Pyomo and the `cbc` solver in Colab.

In [ ]:
# import modules

# this makes beautiful plots using pylab (matplotlib)
%matplotlib inline
from pylab import * # * means import ALL NAME SPACES
import pandas as pd

# useful modules for downloading pyomo onto Colab
import shutil
import sys
import os.path

# install pyomo if it doesn't exist
if not shutil.which("pyomo"):
    !pip install -q pyomo
    assert(shutil.which("pyomo"))

# install the 'cbc' solver if it doesn't exist
if not (shutil.which("cbc") or os.path.isfile("cbc")):
    if "google.colab" in sys.modules:
        !apt-get install -y -qq coinor-cbc
    else:
        try:
            !conda install -c conda-forge coincbc
        except:
            pass

# make sure 'cbc' is the solver that we will invoke later on
assert(shutil.which("cbc") or os.path.isfile("cbc"))

# import ALL NAMESPACES (variables) from pyomo
from pyomo.environ import *

## The problem

Flair makes **tables** and **chairs**.

| | Tables | Chairs | Hours available |
|---|---|---|---|
| Carpentry | 3 | 4 | 2,400 |
| Painting | 2 | 1 | 1,000 |
| **Profit (USD)** | 7 | 5 | - |

Plus: **no more than 450 chairs**, and **at least 100 tables**.

$$Max\ Z = 7T + 5C$$

subject to:
* $3T + 4C \le 2400$ `carpentry`
* $2T + 1C \le 1000$ `painting`
* $C \le 450$ `chair limit`
* $T \ge 100$ `table minimum`
* $T, C \ge 0$ `non-negativity`

# Version A · The hard-coded model

Every Pyomo model has the same pieces, in the same order:

**model → variables → objective → constraints → solve → read the results**

Read the cell below next to the math above. Each line of math became one line of code.

In [ ]:
# you will always need to write this code
model = ConcreteModel()

# declare decision variables
model.tables = Var(domain=NonNegativeReals)
model.chairs = Var(domain=NonNegativeReals)

# declare objective
model.profit = Objective(
                      expr = 7*model.tables + 5*model.chairs, # values come from the table
                      sense = maximize)

# write the constraints one by one, don't forget LHS vs. RHS
# it's OK to have mixed constraints (LHS <= RHS vs. LHS >= RHS)!!!
model.constraint1 = Constraint(expr = 3*model.tables + 4*model.chairs <= 2400) # carpentry hours
model.constraint2 = Constraint(expr = 2*model.tables + 1*model.chairs <= 1000) # painting hours
model.constraint3 = Constraint(expr = model.chairs <= 450) # chair limit
model.constraint4 = Constraint(expr = model.tables >= 100) # table minimum

# show the model you've created
model.pprint()

In [ ]:
#            solver          path the solver
SolverFactory('cbc', executable='/usr/bin/cbc').solve(model).write()

Look for `Termination condition: optimal`. If you see it, you solved! If not, you probably have a
bug (a typo in a constraint, the wrong sense, etc.).

Now read the results. **Don't forget the `()` after everything** - lots of students forget this and
get a bug!

In [ ]:
# the objective function
print("Profit = ", model.profit(), " USD")

# the decision variables
print("Tables = ", model.tables(), " units")
print("Chairs = ", model.chairs(), " units")
print()

# the left-hand side of each constraint - how much did we actually use?
print("Carpentry = ", model.constraint1(), " hours (of 2400)")
print("Painting = ", model.constraint2(), " hours (of 1000)")
print("Chairs = ", model.constraint3(), " units (limit 450)")
print("Tables = ", model.constraint4(), " units (minimum 100)")

**320 tables, 360 chairs, \$4,040.** Same answer as brute force, the graphical method, and
Simplex. Fourth way, same answer.

## Which constraints are binding?

A constraint is **binding** when its left-hand side is sitting right on its limit: there is nothing
left over. Binding constraints are the **natural bottlenecks** in the operation. They are what is
stopping you from making more money.

What is left over is called the **slack**. Pyomo will tell you the slack of any constraint:

- `uslack()` is the room left below an **upper** limit (for a `<=` constraint).
- `lslack()` is the room left above a **lower** limit (for a `>=` constraint).

A constraint only has a limit on one side, and Pyomo reports the other side as infinity, so we take
the smaller of the two. If the slack is zero, the constraint is binding.

The cell below puts the four constraints in a list and checks each one.

In [ ]:
# put the four constraints in a list, with a name for each one
names = ["Carpentry hours", "Painting hours", "Chair limit", "Table minimum"]
constraints = [model.constraint1, model.constraint2, model.constraint3, model.constraint4]

for i in range(len(constraints)):
    c = constraints[i]

    # the slack is on whichever side has a limit (the other side is infinity)
    slack = min(c.uslack(), c.lslack())

    if slack < 0.000001:
        print(names[i], ": value =", c(), "| slack =", slack, " <-- BINDING (a bottleneck!)")
    else:
        print(names[i], ": value =", c(), "| slack =", slack)

**Carpentry and painting are binding**: every hour is used up. Those are Flair's bottlenecks, and
that is the corner where those two lines crossed on your graph.

The chair limit has 90 units of slack (the same 90 you found on your Simplex math check), and the
table minimum has 220 to spare. Those two rules are not holding Flair back at all.

### Try it: Scenario A1

The shop adds a shift, so carpentry goes from **2,400 hours to 3,000 hours**.

1. Go up to the hard-coded model, find the `2400`, and change it to `3000`.
2. Re-run the model cell, the solve cell, the results cell, and the binding cell.
3. Write down the new profit: \_\_\_\_\_\_\_\_
4. Which constraints are binding now? \_\_\_\_\_\_\_\_\_\_\_\_\_\_\_\_
5. **Change it back to 2400** and re-run.

That was easy with one number. Now imagine 40 constraints, and the same number buried in six of them.
Or imagine the boss asks you to add a third product: you would have to edit the variables, the
objective, and **every** constraint. That is the problem with hard-coding.

# Version B · The soft-coded model

Same problem. This time the numbers go in **dictionaries**, and the model reads them with **loops**.

It comes in three parts:

1. **THE DATA** - the word problem, as dictionaries. Running it resets everything to the original problem.
2. **THE SCENARIO** - where you change things.
3. **THE MODEL** - six small cells (3a to 3f) that build the model from whatever is in the
   dictionaries, solve it, and print it. **You never edit these.**

In [ ]:
# ---------- 1. THE DATA (run this to reset to the original problem) ----------
products = ["tables", "chairs"]

# profit per unit
profit = {
    "tables" : 7,
    "chairs" : 5
}

# hours each product needs in each department
hours = {
    "carpentry" : {"tables" : 3, "chairs" : 4},
    "painting"  : {"tables" : 2, "chairs" : 1}
}

# hours available this month
total_time = {
    "carpentry" : 2400,
    "painting"  : 1000
}

# marketing rules - only the products that HAVE a limit are listed
max_units = {"chairs" : 450}    # no more than 450 chairs
min_units = {"tables" : 100}    # at least 100 tables

**The scoreboard.** One more dictionary. It starts out empty, and we use it to remember the profit
from every scenario we try.

A dictionary grows when you **assign a value to a new key**. There is no `append`:

```python
scoreboard = {}                           # {}
scoreboard["base"] = 4040                 # {'base': 4040}
scoreboard["B1 more carpentry"] = 4175    # {'base': 4040, 'B1 more carpentry': 4175}
```

The last cell of the model (3f) does exactly that for you: the **key** is the name of the scenario and
the **value** is the profit the solver found. So every scenario you run adds one line to the
scoreboard.

Run the next cell **once**. If you run it again, you wipe the scoreboard clean.

In [ ]:
# an empty scoreboard: scenario name -> profit
scoreboard = {}

In [ ]:
# ---------- 2. THE SCENARIO (change things here) ----------
scenario = "base"

# Scenario B1: the shop adds a shift (remove the # from the next two lines)
# scenario = "B1 more carpentry"
# total_time["carpentry"] = 3000

# Scenario B2: a new product, desks (remove the # from the next five lines)
# scenario = "B2 add desks"
# products.append("desks")
# profit["desks"] = 10
# hours["carpentry"]["desks"] = 4
# hours["painting"]["desks"] = 2

# Scenario B3: your own idea goes here


## How a loop builds an expression: `+=`

In Version A we typed the objective in one go: `7*model.tables + 5*model.chairs`.

In Version B we **grow** it, one product at a time. `obj_expr += something` means *"take what
`obj_expr` already is, and add `something` to it."* Start from zero, and every pass through the loop
tacks on one more product:

```
start          obj_expr = 0
after tables   obj_expr = 7*x[tables]
after chairs   obj_expr = 7*x[tables] + 5*x[chairs]
```

It grows like a triangle. With two products it is a small triangle, but the same loop would grow 77
rows for 77 products, and you would not have to type any of them.

The next cell is just a demonstration (it is not the model yet). Run it and watch the expression grow.

In [ ]:
# a demonstration of +=   (not the model yet - just watch the expression grow)
demo = ConcreteModel()
demo.x = Var(products, domain=NonNegativeReals)

obj_expr = 0
print("start :", obj_expr)

for product in products:
    obj_expr += profit[product] * demo.x[product]
    print("after", product, ":", obj_expr)

## 3 · The model, one piece at a time

**3a · The model and the decision variables.** One indexed variable, `model.x`, with one entry per
product: `model.x["tables"]`, `model.x["chairs"]`, and so on.

In [ ]:
# ---------- 3a. the model and the decision variables ----------
model = ConcreteModel()

# ONE indexed variable: model.x["tables"], model.x["chairs"], ...
model.x = Var(products, domain=NonNegativeReals)

**3b · The objective.** The same `+=` loop you just watched: profit × units, one product at a time.

In [ ]:
# ---------- 3b. the objective ----------
obj_expr = 0
for product in products:
    obj_expr += profit[product] * model.x[product]

model.profit = Objective(
                      expr = obj_expr,
                      sense = maximize)

print("Maximize:", obj_expr)

**3c · The constraints.** We start an empty `ConstraintList()` and add to it.

- The **department loop** has the same shape as the objective loop: start at zero, add hours × units
  for each product, then say the total has to fit in the hours available.
- The **marketing rules** loop over `max_units` and `min_units`. Those dictionaries only list the
  products that have a limit, so the loop only visits those. We never had to say "tables have no
  maximum" or "chairs have no minimum".

We also keep a list of plain-English names, in the same order, so we can read the results later.

In [ ]:
# ---------- 3c. the constraints ----------
model.constraints = ConstraintList()
constraint_names = []

# one constraint per department
for dept in total_time:
    dept_expr = 0
    for product in products:
        dept_expr += hours[dept][product] * model.x[product]
    model.constraints.add(dept_expr <= total_time[dept])
    constraint_names.append(dept + " hours")

# the marketing rules
for product in max_units:
    model.constraints.add(model.x[product] <= max_units[product])
    constraint_names.append(product + " maximum")
for product in min_units:
    model.constraints.add(model.x[product] >= min_units[product])
    constraint_names.append(product + " minimum")

print(constraint_names)

**3d · Look at what we built.** A `ConstraintList` does not have names like `constraint1`. It just
**numbers** the constraints 1, 2, 3, 4 in the order we added them. `model.pprint()` shows everything:
the variables, the objective, and the numbered constraints. Compare it with the `pprint()` from
Version A. It is the same model.

In [ ]:
# ---------- 3d. show the model ----------
model.pprint()

**3e · Solve it.**

In [ ]:
# ---------- 3e. solve ----------
SolverFactory('cbc', executable='/usr/bin/cbc').solve(model).write()

**3f · Read the results.** The objective, the decision variables, then every constraint by number,
with its slack. Same binding check as Version A, but now it is a loop over the list.

In [ ]:
# ---------- 3f. the results ----------
print("SCENARIO:", scenario)

# the objective function
print("Profit = ", model.profit(), " USD")

# the decision variables
for product in products:
    print(product, " = ", model.x[product](), " units")
print()

# the constraints are numbered 1, 2, 3 ... in the order we added them
k = 1
for name in constraint_names:
    c = model.constraints[k]
    slack = min(c.uslack(), c.lslack())
    if slack < 0.000001:
        print("constraint", k, "-", name, ": value =", c(), "| slack =", slack, " <-- BINDING (a bottleneck!)")
    else:
        print("constraint", k, "-", name, ": value =", c(), "| slack =", slack)
    k += 1

# add this scenario to the scoreboard dictionary: scoreboard[key] = value
scoreboard[scenario] = model.profit()
print()
print("Scoreboard so far:", scoreboard)

Same answer: **320 tables, 360 chairs, \$4,040**, with carpentry and painting binding.

But look back at cells 3a to 3f: there is **not a single number from the word problem in them.** No
2400, no 7, no 450. They do not even know how many products there are. Everything comes from the data.

### Try it: Scenarios B1 and B2

For each scenario:

1. Run **THE DATA** cell (this resets the problem; it does not touch your scoreboard).
2. In **THE SCENARIO** cell, remove the `#` from that scenario's lines.
3. Click on the scenario cell and choose **Runtime → Run cell and below** (Ctrl+F10). That runs the
   scenario and every model cell after it.
4. Put the `#` back when you are done.

| Scenario | What changes | Profit | What did the plan look like? | What is binding? |
|---|---|---|---|---|
| base | nothing | \$4,040 | 320 tables, 360 chairs | carpentry, painting |
| B1 | carpentry hours go to 3,000 | | | |
| B2 | add desks (profit \$10, carpentry 4 hours, painting 2 hours) | | | |
| B3 | your own idea | | | |

Notice what you did for B2: you added a whole new product by adding **four lines of data**. You did
not touch the model. Scroll up and look at the `+=` demonstration and the `pprint()`: the triangle
grew a third row, and every constraint picked up a desks term, all by themselves. Try doing that to
Version A!

In [ ]:
# the scoreboard so far
print(scoreboard)

## Hard-coded vs. soft-coded

| | Hard-coded (Version A) | Soft-coded (Version B) |
|---|---|---|
| Where the numbers live | inside the model | in dictionaries, above the model |
| Easy to read next to the math? | **yes** - one line of math, one line of code | takes a minute to see the loops |
| Change one number | hunt for it in the model | change it in one place |
| Add a product or a department | rewrite the variables, objective and constraints | add a few lines of data |
| 77 products instead of 2 | no thanks | same code |

**Both are fine.** Hard-code when you are learning or the model is tiny. Soft-code when the model is
big, when the data will change, or when someone else has to use your work. In the next notebook the
model has 77 decision variables, and the code is no longer than this one.

## Bottom line

- The Pyomo recipe never changes: **model → variables → objective → constraints → solve → read the
  results.**
- **Binding constraints** (zero slack) are the bottlenecks. Change the data and the bottleneck can move.
- `+=` in a loop grows an expression one term at a time. A **ConstraintList** plus a loop lets you
  write a constraint once and use it for every department.
- **Soft-code the data** and the model does not change when the data does.

**Before you move on:** File → Save a copy in GitHub.